In [2]:
#importing necessary libraries
import pandas as pd
import numpy as np

Step 1: The Cleanup
The content team forgot to log the duration for 'C03' (it's NaN). Fill that missing value in the movies DataFrame with 110.0.

Step 2: The Master Merge
Create your master_df by starting with the history table. Do a left merge to bring in the users table. Then, take that result and do another left merge to bring in the movies table.

Step 3: Feature Engineering
We need to know if people are actually finishing what they start. Create a new column in master_df called Completion_Pct. Calculate it by dividing Watched_Mins by Duration_Mins, and then multiplying by 100.

Step 4: The Automation Loop
Write a for loop that iterates through the unique plans ('Basic', 'Premium') in master_df.
(Hint: Remember your brilliant trick from last time: master_df['Plan'].dropna().unique() to avoid looping over the missing user!)
Inside the loop, filter the master_df to create a temporary DataFrame just for that plan.

Step 5: The Summary (Inside the Loop)
Still inside your loop (indented!), take your filtered plan DataFrame and group it by Genre. Use .agg() to find:

The average (mean) Completion_Pct

The total (sum) Watched_Mins
Make sure to .reset_index() and print the result so you can see it!

In [3]:
users = pd.DataFrame({
    'User_ID': [101, 102, 103, 104],
    'Plan': ['Basic', 'Premium', 'Basic', 'Premium']
})

# Table 2: Content Library (Notice the missing duration for C03!)
movies = pd.DataFrame({
    'Content_ID': ['C01', 'C02', 'C03', 'C04'],
    'Genre': ['Action', 'Comedy', 'Action', 'Drama'],
    'Duration_Mins': [120.0, 90.0, np.nan, 150.0]
})

# Table 3: Watch History
history = pd.DataFrame({
    'Log_ID': [1, 2, 3, 4, 5],
    'User_ID': [101, 102, 101, 104, 999], # User 999 canceled and isn't in our DB!
    'Content_ID': ['C01', 'C03', 'C02', 'C01', 'C04'],
    'Watched_Mins': [60.0, 110.0, 90.0, 120.0, 15.0]
})

In [4]:
#cleean Data
movies['Duration_Mins']=movies['Duration_Mins'].fillna(110.0)

In [5]:
#merge the data
master_df=pd.merge(history,users, on='User_ID', how='left')

master_df=pd.merge(master_df,movies, on='Content_ID', how='left')

In [6]:
master_df

,Log_ID,User_ID,Content_ID,Watched_Mins,Plan,Genre,Duration_Mins
0,1,101,C01,60.0,Basic,Action,120.0
1,2,102,C03,110.0,Premium,Action,110.0
2,3,101,C02,90.0,Basic,Comedy,90.0
3,4,104,C01,120.0,Premium,Action,120.0
4,5,999,C04,15.0,NaN,Drama,150.0


In [7]:
master_df['Completion_pct']=master_df['Watched_Mins']/master_df['Duration_Mins']*100

In [8]:
master_df

,Log_ID,User_ID,Content_ID,Watched_Mins,Plan,Genre,Duration_Mins,Completion_pct
0,1,101,C01,60.0,Basic,Action,120.0,50.0
1,2,102,C03,110.0,Premium,Action,110.0,100.0
2,3,101,C02,90.0,Basic,Comedy,90.0,100.0
3,4,104,C01,120.0,Premium,Action,120.0,100.0
4,5,999,C04,15.0,NaN,Drama,150.0,10.0


In [9]:
for plan_type in master_df['Plan'].dropna().unique():
    plan_df=master_df[master_df['Plan']==plan_type]

In [10]:
summary_df=master_df.groupby('Genre').agg({
    'Completion_pct':'mean',
    'Watched_Mins':'sum'
}).reset_index()

In [11]:
summary_df

,Genre,Completion_pct,Watched_Mins
0,Action,83.333333,290.0
1,Comedy,100.000000,90.0
2,Drama,10.000000,15.0
